# Week 8 Lab: Monte Carlo Simulation of a Schooling Model

## Learning Objectives

By the end of this lab, you will be able to:
1. Calibrate a stylized economic model using real data
2. Write data generating processes (DGPs) in Julia
3. Conduct Monte Carlo simulations to study estimator properties
4. Understand the finite-sample vs. asymptotic behavior of OLS

## The Idea: A Different Perspective on Regression

In typical empirical work, you start with a **data set** and estimate regression coefficients. The true parameter values are unknown, and you hope your estimator gets close to them. You can compute standard errors and confidence intervals, but you never actually *know* whether your particular estimate is close to the truth or not.

**Monte Carlo simulation flips this around.** Instead of starting with data and estimating unknown parameters, we:

1. **Choose the true parameter values ourselves** (we play the role of nature)
2. **Generate artificial data** according to a model with those parameters
3. **Pretend we don't know the true values** and estimate them using our standard methods
4. **Compare our estimates to the truth** (which we *do* know, since we chose it!)

This lets us ask questions that are impossible to answer with real data: *How close are our estimates to the true values? How often do our confidence intervals actually contain the truth? What happens when key assumptions are violated?*

### The Data Generating Process (DGP)

The **DGP** is the complete specification of how the data are generated. It includes the functional forms, the distributions of random variables, and the true parameter values. Think of it as a recipe that, given some random draws, produces a data set.

When we write a DGP in code, we create a function that takes parameter values as inputs and returns simulated data as output.

### Calibration: Making the DGP Realistic

A DGP is only useful if it's **realistic**. If we simulate data where everyone has 50 years of schooling and earns \$1 million, our conclusions won't apply to actual labour markets.

**Calibration** means choosing parameter values so that the simulated data *look like* real data. We do this by matching key features: the simulated mean years of schooling should be similar to what we observe in reality; the variance of log wages should be plausible; the regression coefficients should be in a realistic range.

In this lab, we'll calibrate our DGP using Card's (1993) data, so our simulations produce data with similar characteristics to the real world.

### Why This Matters

Econometric theory tells us about **asymptotic** properties (as $N \to \infty$): consistency ($\hat{\beta} \xrightarrow{p} \beta$) and asymptotic normality. But in practice, we have **finite samples**. Monte Carlo simulation lets us verify that the asymptotic approximations work reasonably well for realistic sample sizes, study the actual bias and variance of our estimators, and explore what happens when assumptions fail (which we'll do next week when we introduce endogeneity).

## Our DGP: A Simplified Schooling Model

We'll use a simplified version of Card's schooling model. The DGP has the following structure:

$$\begin{align}
u &\sim N(0, \sigma_u^2) \\
A &\sim N(0, \sigma_A^2) && \text{(unobserved ability)} \\
S &= \pi + A && \text{(schooling)} \\
Y &= \beta_1 + \beta_2 S + \beta_3 A + u && \text{(log earnings)}
\end{align}$$

The parameters are: $\beta_1$ (intercept), $\beta_2$ (return to schooling), $\beta_3$ (direct effect of ability on earnings), $\pi$ (mean schooling), $\sigma_u^2$ (idiosyncratic variance), and $\sigma_A^2$ (ability variance).

### Key Features

1. **Ability affects both schooling and earnings**: More able people get more education ($S = \pi + A$) and earn more ($\beta_3 A$ in wage equation)

2. **When $\beta_3 \neq 0$**: OLS will be biased because $\text{Cov}(S, u + \beta_3 A) = \beta_3 \cdot \text{Var}(A) \neq 0$

3. **When $\beta_3 = 0$**: No endogeneity, OLS is unbiased

### The Econometrician's Problem

In our simulation, **we know all the parameters**. But the econometrician observes only $(S_i, Y_i)$, **not** ability $A_i$. From their perspective, they estimate:
$$Y_i = \beta_1 + \beta_2 S_i + \varepsilon_i$$

where $\varepsilon_i = \beta_3 A_i + u_i$ is the composite error. The econometrician doesn't know the true $\beta_2$—but we do! This is the power of simulation.

In [ ]:
using LinearAlgebra, Printf, Distributions, Random, Plots, DelimitedFiles, Statistics

# Load functions from earlier weeks
include("emet_8014_functions.jl")

## Exercise 1: Calibrate the Model Using Card's Data

As discussed above, **calibration** means choosing parameter values so the simulated data match key features of real data. We want our artificial "laboratories" to resemble the real world.

We'll calibrate using Card's (1993) data, setting $\beta_3 = 0$ initially (no endogeneity). This gives us a baseline where OLS should work well.

The following table shows how to calibrate each parameter. The idea is to use observable moments from the data (means, variances, regression coefficients) to pin down the DGP parameters:

(Note: you can round crudely to obtain simple calibrated values.)

|                           | Use the card data set for your calibration          | Calibrated values  |
|---------------------------|-------------------------------------------------------------------------------|--------------------|
| $\beta_1$                 | OLS estimator $\widehat{\beta}_1$                                             | ???               |
| $\beta_2$                 | OLS estimator $\widehat{\beta}_2$                                             | ???               |
| $\pi$                     | sample average of schooling                                                   | ???               |
| $\sigma_u^2$              | conditional variance of log wages for people with 12 years of schooling       | ???               |
| $\sigma_A^2$              | sample variance of schooling                                                  | ???               |

Enter your calibrated values in the last column.

In [ ]:
# Load Card's data to calibrate our model
data = readdlm("../datasets/card.csv", ',')

# Extract relevant variables
# Column 33 is log wages, column 4 is education
Y_card = Vector{Float64}(data[:, 33])  # log wages
S_card = Vector{Float64}(data[:, 4])   # education
N_card = length(Y_card)

# Create design matrix for simple regression
X_card = nothing  # YOUR CODE HERE

# OLS estimates using lm_ols from emet_8014_functions.jl
# Hint: lm_ols(Y, X) returns (coefficients, covariance_matrix)
β_card = nothing  # YOUR CODE HERE
Ω_card = nothing  # YOUR CODE HERE

@printf "OLS estimates from Card's data (simple regression):\n"
@printf "  β₁ (intercept): %.2f\n" β_card[1]
@printf "  β₂ (return to schooling): %.4f\n" β_card[2]

In [ ]:
# Additional calibration targets

# π = E(S): mean years of schooling
π_calib = nothing  # YOUR CODE HERE
@printf "\nMean schooling (π): %.2f years\n" π_calib

# σ²_A = Var(S): since S = π + A, Var(S) = Var(A) = σ²_A
σ2_A_calib = nothing  # YOUR CODE HERE
@printf "Variance of schooling (σ²_A): %.2f\n" σ2_A_calib

# σ²_u: conditional variance of log wages for people with 12 years of schooling
# This is an approximation to Var(u)
# Hint: subset the data appropriately, then compute variance
Y_12 = nothing  # YOUR CODE HERE
σ2_u_calib = nothing  # YOUR CODE HERE
@printf "Conditional variance of log wages | S=12 (σ²_u): %.2f\n" σ2_u_calib

In [ ]:
# Summary of calibrated parameters
println("\n" * "="^50 * "\n")
@printf "CALIBRATED PARAMETER VALUES\n"
println("="^50 * "\n\n")
@printf "%-15s %10s %15s\n" "Parameter" "Symbol" "Value"
println("-"^50 * "\n")
@printf "%-15s %10s %15.2f\n" "Intercept" "β₁" β_card[1]
@printf "%-15s %10s %15.4f\n" "Return to educ" "β₂" β_card[2]
@printf "%-15s %10s %15.2f\n" "Ability effect" "β₃" 0.0
@printf "%-15s %10s %15.2f\n" "Mean schooling" "π" π_calib
@printf "%-15s %10s %15.2f\n" "Error variance" "σ²_u" σ2_u_calib
@printf "%-15s %10s %15.2f\n" "Ability variance" "σ²_A" σ2_A_calib
println("-"^50 * "\n")

## Exercise 2: Write the DGP Function

Now we create a function that generates random samples from our calibrated DGP.

Write a function `schooling_sample` that takes

* two arguments true coefficient $\beta_2$ and sample size $N$;

* keyword arguments for $\pi$, $\beta_1$, $\beta_3$, $\sigma_u^2$, and $\sigma_A^2$ (set to equal your calibrated values)

    see https://julia.quantecon.org/getting_started_julia/julia_essentials.html#optional-and-keyword-arguments

and returns the random sample `S` and `Y` following the above DGP. (Note: `Y` is the **logarithm** of wages.)

The function should:
1. Draw `u` from $N(0, \sigma_u^2)$
2. Draw `A` from $N(0, \sigma_A^2)$
3. Compute $S = \pi + A$
4. Compute $Y = \beta_1 + \beta_2 S + \beta_3 A + u$
5. Return `(S, Y)`

In [ ]:
"""
    schooling_sample(β2, N; β1=β_card[1], β3=0.0, π=π_calib, σ2_u=σ2_u_calib, σ2_A=σ2_A_calib, seed=nothing)

Generate a random sample from the schooling DGP.

The DGP is:
    u ~ N(0, σ²_u)
    A ~ N(0, σ²_A)  (unobserved ability)
    S = π + A        (schooling)
    Y = β₁ + β₂S + β₃A + u  (log earnings)

# Arguments
- `β2`: Return to schooling (main parameter of interest)
- `N`: Sample size
- `β1`: Intercept (default calibrated to Card's data)
- `β3`: Effect of ability on earnings (β₃ ≠ 0 creates endogeneity)
- `π`: Mean years of schooling
- `σ2_u`: Variance of idiosyncratic error
- `σ2_A`: Variance of ability (= variance of schooling)
- `seed`: Random seed for reproducibility (nothing = no seed)

# Returns
- `S`: Vector of schooling values
- `Y`: Vector of log earnings
"""
function schooling_sample(β2::Real, N::Integer;
                          β1::Real=β_card[1], β3::Real=0.0,
                          π::Real=π_calib, σ2_u::Real=σ2_u_calib, σ2_A::Real=σ2_A_calib,
                          seed::Union{Integer,Nothing}=nothing)
    # Set seed if provided
    if seed !== nothing
        Random.seed!(seed)
    end
    
    # Draw random components
    # Hint: the Distributions package uses standard deviation, not variance
    u = nothing  # YOUR CODE HERE: Idiosyncratic error
    A = nothing  # YOUR CODE HERE: Unobserved ability
    
    # Generate observables following the DGP equations above
    S = nothing  # YOUR CODE HERE
    Y = nothing  # YOUR CODE HERE
    
    return S, Y
end

## Exercise 3: Single Sample Estimation

Let's verify our DGP works by drawing a sample and estimating $\beta_2$ via OLS.

In [ ]:
# Generate a sample of size 100
S, Y = schooling_sample(0.07, 100, seed=25) # play around with different seeds

# Estimate by OLS
N = length(Y)
X = nothing  # YOUR CODE HERE: Design matrix
β_hat = nothing  # YOUR CODE HERE: OLS coefficients

@printf "Sample size: N = %d\n" N
@printf "True β₂ = 0.07\n"
@printf "OLS estimate β̂₂ = %.4f\n" β_hat[2]

## Exercise 4: Large Sample Estimation

With a much larger sample, OLS should be closer to the true value (consistency).

In [ ]:
# Generate a much larger sample
S_large, Y_large = schooling_sample(0.07, 100_000, seed=25)

N_large = length(Y_large)
X_large = nothing  # YOUR CODE HERE
β_hat_large = nothing  # YOUR CODE HERE

@printf "Sample size: N = %d\n" N_large
@printf "True β₂ = 0.07\n"
@printf "OLS estimate β̂₂ = %.6f\n" β_hat_large[2]
@printf "\nThe estimate is much closer to 0.07 with more data (consistency).\n"

## Exercise 5: Monte Carlo Simulation

Now we put it all together. We'll generate **many** samples from our calibrated DGP and study the **distribution** of $\hat{\beta}_2$ across these samples.

This is where the "role reversal" pays off. Because we *know* the true $\beta_2 = 0.07$, we can:
1. **Verify unbiasedness**: Is the average of our estimates equal to the true value? That is, is $E(\hat{\beta}_2) = \beta_2$?
2. **Study variance**: How spread out are the estimates? Does the spread shrink with sample size?
3. **Check the CLT approximation**: Does the histogram of estimates look like a normal distribution?

In [ ]:
"""
    monte_carlo_ols(β2::Real, N::Integer, R::Integer; kwargs...)

Run Monte Carlo simulation of OLS estimation.

# Arguments
- `β2`: True value of return to schooling
- `N`: Sample size for each replication
- `R`: Number of Monte Carlo replications
- `kwargs...`: Additional arguments passed to schooling_sample

# Returns
- Vector of R OLS estimates of β₂
"""
function monte_carlo_ols(β2::Real, N::Integer, R::Integer; kwargs...)
    β2_hat_values = Vector{Float64}(undef, R)
    
    for r in 1:R
        # Generate sample with different seed for each replication
        S, Y = schooling_sample(β2, N; seed=r, kwargs...)
        
        # Estimate OLS and store the slope coefficient
        X = nothing  # YOUR CODE HERE
        β_hat = nothing  # YOUR CODE HERE
        
        β2_hat_values[r] = β_hat[2]
    end
    
    return β2_hat_values
end

In [ ]:
# Run Monte Carlo for different sample sizes
R = 1000  # Number of replications
β2_true = 0.07
sample_sizes = [30, 100, 1000, 10_000]

plots = Plots.Plot[]

for N in sample_sizes
    β2_hat_mc = monte_carlo_ols(β2_true, N, R)
    
    # Create histogram
    p = histogram(β2_hat_mc, 
                  normalize=:pdf,
                  alpha=0.7,
                  label="MC distribution",
                  title="N = $N",
                  xlabel="β̂₂")
    
    # Add vertical line at true value
    vline!(p, [β2_true], 
           label="True β₂ = $β2_true",
           linewidth=2,
           color=:red)
    
    push!(plots, p)
end

plot(plots..., layout=(2, 2), size=(900, 700))

### Observations from Monte Carlo

1. **Unbiasedness**: The distribution is centered around $\beta_2 = 0.07$ (OLS is unbiased when $\beta_3 = 0$)

2. **Variance decreases with N**: The spread of estimates shrinks as sample size grows

3. **CLT kicks in**: Even for small N, the distribution looks approximately normal

## Exercise 6: Comparing to Asymptotic Distribution

From lecture, the asymptotic distribution of $\hat{\beta}_2^{OLS}$ is:

$$\hat{\beta}_2 \overset{\text{approx}}{\sim} N\left(\beta_2, \frac{\sigma_u^2}{N \cdot \sigma_A^2}\right)$$

Let's overlay this theoretical distribution on our Monte Carlo histograms.

In [ ]:
# Parameters for asymptotic distribution
σ2_u = σ2_u_calib
σ2_A = σ2_A_calib

plots = Plots.Plot[]

for N in sample_sizes
    β2_hat_mc = monte_carlo_ols(β2_true, N, R)
    
    # Asymptotic standard error
    # Hint: use the formula from the markdown cell above
    se_asymp = nothing  # YOUR CODE HERE
    
    # Create histogram
    p = histogram(β2_hat_mc,
                  normalize=:pdf,
                  alpha=0.7,
                  label="Monte Carlo",
                  title="N = $N (SE = $(round(se_asymp, digits=4)))",
                  xlabel="β̂₂")
    
    # Overlay asymptotic normal distribution
    x_range = range(β2_true - 4*se_asymp, β2_true + 4*se_asymp, length=200)
    asymp_dist = Normal(β2_true, se_asymp)
    plot!(p, x_range, pdf.(asymp_dist, x_range),
          label="Asymptotic N",
          linewidth=2,
          color=:red)
    
    push!(plots, p)
end

plot(plots..., layout=(2, 2), size=(900, 700))

### Key Insight

The asymptotic normal approximation works well even for moderate sample sizes!

This justifies using normal-based inference (t-tests, confidence intervals) in practice.

## Summary

### What We Learned

1. **Calibration**: How to set DGP parameters to match real data
   - Used Card's data to calibrate $\beta_1, \beta_2, \pi, \sigma^2_u, \sigma^2_A$

2. **DGP Implementation**: Created a flexible `schooling_sample` function
   - Keyword arguments allow easy modification of parameters
   - The $\beta_3$ parameter controls endogeneity

3. **Monte Carlo Simulation**: 
   - Generate many samples and estimate each one
   - Study the distribution of the estimator
   - Verify theoretical predictions (unbiasedness, consistency, CLT)

4. **Asymptotic Approximation**:
   - The normal approximation is remarkably accurate even for N = 30
   - Variance of $\hat{\beta}_2$ is $\sigma^2_u / (N \cdot \sigma^2_A)$

### Looking Ahead

In Week 9, we'll:
- Introduce endogeneity ($\beta_3 \neq 0$) and see OLS bias
- Study the power of hypothesis tests
- Understand how sample size affects statistical power